In [ ]:
import subprocess, sys

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.49.0',
    'peft==0.14.0',
    'trl==0.15.0',
    'accelerate==1.3.0',
    'bitsandbytes==0.45.2',
    'qwen-vl-utils==0.0.10',
    'datasets',
], check=True)

In [ ]:
import os
import torch

os.environ['CUDA_VISIBLE_DEVICES'] = '0'

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    USE_BF16 = 'A100' in gpu_name or 'H100' in gpu_name
    print(f'GPU: {gpu_name}  |  VRAM: {vram:.1f} GB  |  Precision: {"bf16" if USE_BF16 else "fp16"}')
else:
    raise RuntimeError('No GPU found. Enable GPU in Kaggle: Settings -> Accelerator -> GPU T4 x2')

In [ ]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for f in filenames:
        print(os.path.join(dirname, f))

In [ ]:
import json, requests, zipfile, random
from PIL import Image
from tqdm import tqdm
from datasets import Dataset

ZIP_PATH    = '/kaggle/working/Dataset.zip'
EXTRACT_DIR = '/kaggle/working/KIDO_images'

if not os.path.exists(ZIP_PATH):
    url = 'https://huggingface.co/datasets/serdarciftci/KIDO/resolve/main/Dataset.zip'
    r = requests.get(url, stream=True)
    total = int(r.headers.get('content-length', 0))
    downloaded = 0
    with open(ZIP_PATH, 'wb') as f:
        for chunk in r.iter_content(chunk_size=2*1024*1024):
            f.write(chunk)
            downloaded += len(chunk)
            print(f'\r  {downloaded/1024/1024:.0f} / {total/1024/1024:.0f} MB', end='', flush=True)
    print()

if not os.path.exists(EXTRACT_DIR):
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_DIR)

all_images = []
for root, _, files in os.walk(EXTRACT_DIR):
    for f in files:
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            all_images.append(os.path.join(root, f))
all_images.sort()
print(f'Found {len(all_images)} images')

GRADES_PATH = '/kaggle/input/datasets/divitsharma21/graded-dataset/graded_dataset.json'
if not os.path.exists(GRADES_PATH):
    GRADES_PATH = '/kaggle/working/graded_dataset.json'
    if not os.path.exists(GRADES_PATH):
        raise FileNotFoundError(
            'graded_dataset.json not found.\n'
            'Add it as a Kaggle dataset input or copy to /kaggle/working/'
        )

with open(GRADES_PATH, 'r') as f:
    raw_grades = json.load(f)
print(f'Loaded {len(raw_grades)} grade records')

In [ ]:
SYSTEM_PROMPT = (
    'You are an expert art teacher evaluating children\'s hand-drawn sketches. '
    'Grade each drawing on four criteria and provide a structured JSON assessment.'
)

USER_PROMPT = (
    'Please grade this children\'s sketch on the following criteria:\n'
    '- Clarity (1-5): How clearly the subject is recognizable\n'
    '- Detail (1-5): Level of artistic detail and effort\n'
    '- Creativity (1-5): Originality and imaginative elements\n'
    '- Mood (one word): The emotional tone of the drawing\n\n'
    'Provide a brief analysis then output your grades as JSON.'
)

def mood_desc(m):
    tones = {
        'Playful': 'energetic and fun', 'Sad': 'melancholic', 'Happy': 'joyful',
        'Angry': 'intense', 'Calm': 'peaceful', 'Excited': 'enthusiastic',
        'Mysterious': 'intriguing', 'Dark': 'somber', 'Cheerful': 'bright and uplifting'
    }
    return tones.get(str(m), 'expressive')

def build_response(r):
    c, d, cr, mood = r['clarity'], r['detail'], r['creativity'], r['mood']
    return (
        f"Looking at this drawing, here is my assessment:\n\n"
        f"**Clarity ({c}/5):** The subject is {'clearly' if c >= 4 else 'somewhat' if c >= 3 else 'not clearly'} "
        f"recognizable, with {'strong' if c >= 4 else 'moderate' if c >= 3 else 'limited'} visual communication.\n"
        f"**Detail ({d}/5):** The level of detail is {'high' if d >= 4 else 'moderate' if d >= 3 else 'basic'}, "
        f"showing {'significant' if d >= 4 else 'some' if d >= 3 else 'minimal'} effort on fine elements.\n"
        f"**Creativity ({cr}/5):** The artistic approach is {'highly imaginative' if cr >= 4 else 'creative' if cr >= 3 else 'straightforward'}, "
        f"demonstrating {'strong' if cr >= 4 else 'decent' if cr >= 3 else 'conventional'} originality.\n"
        f"**Mood:** The drawing has a {mood_desc(mood)} quality, conveying a sense of {mood.lower() if isinstance(mood, str) else mood}.\n\n"
        f"Final grades:\n"
        f"{json.dumps({'clarity': c, 'detail': d, 'creativity': cr, 'mood': mood})}"
    )

data_list = []
skipped = 0

for record in tqdm(raw_grades, desc='Building dataset'):
    idx = record.get('image_index', -1)
    if idx < 0 or idx >= len(all_images):
        skipped += 1; continue
    if any(record.get(k) is None for k in ['clarity', 'detail', 'creativity', 'mood']):
        skipped += 1; continue
    try:
        Image.open(all_images[idx]).convert('RGB')
    except:
        skipped += 1; continue

    data_list.append({
        'image_path': all_images[idx],
        'assistant_text': build_response(record),
        'image_index': idx,
        'true_grades': {k: v for k, v in record.items() if k != 'image_index'},
    })

random.seed(42)
random.shuffle(data_list)
split = int(len(data_list) * 0.8)
train_list = data_list[:split]
test_list  = data_list[split:]

print(f'Train: {len(train_list)}  |  Test: {len(test_list)}  |  Skipped: {skipped}')

In [ ]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from peft import LoraConfig, get_peft_model, TaskType

MODEL_ID   = 'Qwen/Qwen2-VL-2B-Instruct'
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 512 * 28 * 28

processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
)
model.enable_input_require_grads()

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

tok          = processor.tokenizer
VISION_START = tok.convert_tokens_to_ids('<|vision_start|>')
VISION_END   = tok.convert_tokens_to_ids('<|vision_end|>')
IMAGE_PAD    = tok.convert_tokens_to_ids('<|image_pad|>')
IM_START     = tok.convert_tokens_to_ids('<|im_start|>')
PAD_ID       = tok.pad_token_id

In [ ]:
from qwen_vl_utils import process_vision_info

IGNORE             = -100
IMAGE_SPECIAL_IDS  = {VISION_START, VISION_END, IMAGE_PAD}
ASSISTANT_HEADER_TOKENS = tok.encode('assistant', add_special_tokens=False)


def find_assistant_response_start(ids_list):
    n      = len(ids_list)
    ah_len = len(ASSISTANT_HEADER_TOKENS)
    for i in range(n - 1, -1, -1):
        if ids_list[i] == IM_START:
            if ids_list[i+1 : i+1+ah_len] == ASSISTANT_HEADER_TOKENS:
                return i + 1 + ah_len + 1
    return -1


def collate_fn(examples):
    texts, images_batch = [], []

    for ex in examples:
        messages = [
            {'role': 'system',    'content': [{'type': 'text',  'text': SYSTEM_PROMPT}]},
            {'role': 'user',      'content': [{'type': 'image', 'image': ex['image_path']},
                                               {'type': 'text',  'text': USER_PROMPT}]},
            {'role': 'assistant', 'content': [{'type': 'text',  'text': ex['assistant_text']}]},
        ]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        image_inputs, _ = process_vision_info(messages)
        texts.append(text)
        images_batch.append(image_inputs)

    flat_images = [img for imgs in images_batch for img in (imgs or [])]
    batch = processor(
        text=texts,
        images=flat_images if flat_images else None,
        return_tensors='pt',
        padding=True,
    )

    labels = batch['input_ids'].clone()

    labels[labels == PAD_ID] = IGNORE
    for img_tok_id in IMAGE_SPECIAL_IDS:
        labels[labels == img_tok_id] = IGNORE

    for i in range(labels.shape[0]):
        ids_list   = batch['input_ids'][i].tolist()
        resp_start = find_assistant_response_start(ids_list)
        if resp_start > 0:
            labels[i, :resp_start] = IGNORE
        else:
            labels[i] = IGNORE

    batch['labels'] = labels
    return batch


test_batch   = collate_fn([train_list[0]])
labels_s     = test_batch['labels'][0]
active_mask  = labels_s != IGNORE
n_active     = active_mask.sum().item()
n_total      = len(labels_s)
decoded      = tok.decode(labels_s[active_mask].tolist(), skip_special_tokens=True)

print(f'Active tokens: {n_active}/{n_total} ({100*n_active/n_total:.1f}%)')
print(decoded[:300])

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback
from datasets import Dataset

hf_train = Dataset.from_list(train_list)


class LossMonitor(TrainerCallback):
    def __init__(self):
        self.losses = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and 'loss' in logs:
            loss = logs['loss']
            self.losses.append((state.global_step, loss))
            if state.global_step % 20 == 0 or state.global_step <= 5:
                print(f'Step {state.global_step:4d} | Loss: {loss:.4f}', flush=True)


training_args = SFTConfig(
    output_dir='/kaggle/working/vlm_adapter',
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    num_train_epochs=3,
    learning_rate=1e-4,
    warmup_steps=30,
    lr_scheduler_type='cosine',
    fp16=not USE_BF16,
    bf16=USE_BF16,
    optim='adamw_torch',
    logging_steps=5,
    save_strategy='epoch',
    save_total_limit=2,
    remove_unused_columns=False,
    dataset_kwargs={'skip_prepare_dataset': True},
    max_seq_length=1024,
)

monitor = LossMonitor()

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=hf_train,
    data_collator=collate_fn,
    processing_class=processor,
    callbacks=[monitor],
)

result = trainer.train()
print(f'Final loss: {result.training_loss:.4f}  |  Steps: {result.global_step}')

In [ ]:
import matplotlib.pyplot as plt

SAVE_DIR = '/kaggle/working/vlm_art_grader_final'
trainer.save_model(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print(f'Model saved to {SAVE_DIR}')

if monitor.losses:
    steps, losses = zip(*monitor.losses)
    plt.figure(figsize=(10, 4))
    plt.plot(steps, losses, linewidth=2, color='#2196F3')
    plt.axhline(y=2.35, color='red',   linestyle='--', alpha=0.5, label='Old flatline (2.35)')
    plt.axhline(y=1.5,  color='green', linestyle='--', alpha=0.5, label='Target (<1.5)')
    plt.xlabel('Step')
    plt.ylabel('Loss')
    plt.title('Training Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('/kaggle/working/loss_curve.png', dpi=150)
    plt.show()

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt

model.eval()
model.config.use_cache = True


def grade_image(img_path):
    messages = [
        {'role': 'system', 'content': [{'type': 'text',  'text': SYSTEM_PROMPT}]},
        {'role': 'user',   'content': [{'type': 'image', 'image': img_path},
                                        {'type': 'text',  'text': USER_PROMPT}]},
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, _ = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, return_tensors='pt').to('cuda:0')

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=300, do_sample=False)

    trimmed = out[0][inputs['input_ids'].shape[1]:]
    response = processor.decode(trimmed, skip_special_tokens=True)

    parsed = None
    try:
        m = re.search(r'\{[^{}]+\}', response, re.DOTALL)
        if m:
            parsed = json.loads(m.group())
    except:
        pass

    return response, parsed


samples = random.sample(test_list, min(3, len(test_list)))
for i, s in enumerate(samples):
    response, parsed = grade_image(s['image_path'])
    print(f'--- Sample {i+1} ---')
    print(f'True:      {s["true_grades"]}')
    print(f'Predicted: {parsed}')
    print()

In [ ]:
from collections import defaultdict

errors = defaultdict(list)

for s in tqdm(test_list, desc='Evaluating'):
    _, parsed = grade_image(s['image_path'])
    if parsed is None:
        continue
    true = s['true_grades']
    for key in ['clarity', 'detail', 'creativity']:
        if key in true and key in parsed:
            try:
                errors[key].append(abs(float(parsed[key]) - float(true[key])))
            except:
                pass

mae_vals = {k: np.mean(v) for k, v in errors.items() if v}
overall_mae = np.mean([e for v in errors.values() for e in v])

print('MAE per criterion:')
for k, v in mae_vals.items():
    print(f'  {k}: {v:.3f}')
print(f'Overall MAE: {overall_mae:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(mae_vals.keys(), mae_vals.values(), color=['#3498db', '#e74c3c', '#2ecc71'])
axes[0].set_title('MAE per Criterion')
axes[0].set_ylabel('MAE')
axes[0].grid(True, alpha=0.3)

all_err = [e for v in errors.values() for e in v]
axes[1].hist(all_err, bins=20)
axes[1].set_title('Error Distribution')
axes[1].set_xlabel('Absolute Error')
axes[1].set_ylabel('Count')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/evaluation.png', dpi=150)
plt.show()

In [ ]:
import shutil

package_dir = '/kaggle/working/download_package'
shutil.rmtree(package_dir, ignore_errors=True)
os.makedirs(package_dir, exist_ok=True)

shutil.copy('/kaggle/working/loss_curve.png',     package_dir)
shutil.copy('/kaggle/working/evaluation.png',      package_dir)
shutil.copytree('/kaggle/working/vlm_adapter',    f'{package_dir}/vlm_adapter')
shutil.make_archive('/kaggle/working/vlm_art_grader_files', 'zip', package_dir)

print('Zip ready: /kaggle/working/vlm_art_grader_files.zip')